# Feature Extraction

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, LSTM, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# --- Load data and create windows (if not already loaded) ---
# Check if windows and window_labels already exist
if 'windows' not in globals() or 'window_labels' not in globals():
    # Load data from previous notebook
    data_dir = Path('../data/raw/CAPPIMU/data')
    subject_id = 1
    
    def read_subject_trial(subject_id, data_dir=data_dir):
        subject_path = data_dir / f'subject_{subject_id}'
        if not subject_path.exists():
            raise ValueError(f"Subject {subject_id} data not found")
        
        csv_path = subject_path / 'insole.csv'
        if csv_path.exists():
            return pd.read_csv(csv_path)
        else:
            raise FileNotFoundError(f"Insole data not found: {csv_path}")
    
    # Load insole data
    data = read_subject_trial(subject_id)
    
    # Separate sensor values (exclude timestamp and label columns)
    sensor_cols = [col for col in data.columns if col not in ['time', 'label']]
    sensor_data = data[sensor_cols].values  # shape: (num_samples, num_channels)
    
    # Convert string labels to integers
    label_encoder = LabelEncoder()
    labels = label_encoder.fit_transform(data['label'].values)
    print(f"Label mapping: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")
    
    # Define window parameters
    fs = 100  # sensor frequency in Hz
    window_sec = 1.0  # 1-second window
    window_size = int(fs * window_sec)  # samples per window
    stride = int(window_size * 0.5)  # 50% overlap
    
    # Windowing function
    def create_windows_multi_channel(sensor_data, labels=None, window_size=100, stride=50):
        windows = []
        window_labels = []
        for start in range(0, len(sensor_data) - window_size + 1, stride):
            end = start + window_size
            windows.append(sensor_data[start:end, :])  # multi-channel window
            if labels is not None:
                # Assign window label: majority label in the window
                window_labels.append(np.bincount(labels[start:end]).argmax())
        windows = np.array(windows)  # shape: (num_windows, window_size, num_channels)
        if labels is not None:
            window_labels = np.array(window_labels)
            return windows, window_labels
        return windows
    
    # Create windows
    windows, window_labels = create_windows_multi_channel(sensor_data, labels, window_size, stride)
    print(f"Windows shape: {windows.shape}")
    print(f"Window labels shape: {window_labels.shape}")

# One-hot encode labels for 21 classes
y_onehot = to_categorical(window_labels, num_classes=21)

# Optional: train-test split
X_train, X_test, y_train, y_test = train_test_split(windows, y_onehot, test_size=0.2, random_state=42, stratify=y_onehot)

# Optional: standardize input per channel
num_channels = X_train.shape[2]
for ch in range(num_channels):
    scaler = StandardScaler()
    X_train[:, :, ch] = scaler.fit_transform(X_train[:, :, ch])
    X_test[:, :, ch] = scaler.transform(X_test[:, :, ch])

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test labels shape: {y_test.shape}")



Label mapping: {'brushing_teeth': 0, 'cut': 1, 'cycle': 2, 'drink': 3, 'eat': 4, 'fall': 5, 'folding_clothes': 6, 'hang_out_clothes': 7, 'ironing': 8, 'mop': 9, 'play_phone': 10, 'run': 11, 'sweep': 12, 'use_computer': 13, 'walk': 14, 'wash_dish': 15, 'wash_face': 16, 'wash_window': 17, 'watch_tv': 18, 'wc': 19, 'write': 20}
Windows shape: (4072, 100, 16)
Window labels shape: (4072,)

Training set shape: (3257, 100, 16)
Test set shape: (815, 100, 16)
Training labels shape: (3257, 21)
Test labels shape: (815, 21)


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, BatchNormalization

model = Sequential()

# CNN feature extractor
model.add(Conv1D(64, kernel_size=3, activation='relu', input_shape=(100, 16)))
model.add(BatchNormalization())
model.add(Conv1D(64, kernel_size=3, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))

# Temporal modeling
model.add(LSTM(128))
model.add(Dropout(0.3))

# Classifier
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(21, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


c:\Users\monya\fall-prediction-cappimu\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 98, 64)         │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 98, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 96, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 48, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 21)             │         1,365 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 124,181 (485.08 KB)

 Trainable params: 124,053 (484.58 KB)

 Non-trainable params: 128 (512.00 B)

## Model Input Preparation

- Windowed sensor data shape: (4072, 100, 16)
  - 100 time steps per window (1 s at 100 Hz)
  - 16 sensor channels (FSR + IMU)
- Labels encoded into 21 activity classes using one-hot encoding
- Dataset split into training (80%) and test (20%) sets
- Training data standardized per sensor channel using z-score normalization
- Overlapping windows used (50% overlap)

This processed data is used as input to a CNN–LSTM model for activity classification.